# 03 — Entraînement Word2Vec par tranche temporelle
Un modèle SGNS par élection, avec initialisation t depuis t-1 (Kim et al. 2014).

In [21]:
import pickle
import numpy as np
from pathlib import Path
from gensim.models import Word2Vec

Path('../results/models').mkdir(exist_ok=True)

YEARS = [1973, 1978, 1981, 1988, 1993]
PARAMS = dict(
    vector_size=100,
    window=5,
    min_count=25,    
    sg=1,            
    negative=10,     # SGNS : 10 negative samples (Mikolov 2013)
    sample=1e-4,     # Subsampling des mots fréquents
    epochs=50,
    workers=4,
    seed=42
)

# Charger tous les corpus
corpora = {}
for year in YEARS:
    with open(f'../data/processed/corpus_{year}.pkl', 'rb') as f:
        corpora[year] = pickle.load(f)
    print(f'{year}: {len(corpora[year])} docs, {sum(len(d) for d in corpora[year]):,} tokens')

1973: 3917 docs, 1,168,211 tokens
1978: 5025 docs, 1,504,300 tokens
1981: 3179 docs, 774,182 tokens
1988: 3615 docs, 740,342 tokens
1993: 5929 docs, 1,468,404 tokens


In [22]:
# Whitelist curée : on applique UNIQUEMENT des bigrammes politiquement attestés
# Note sur les lemmes : le corpus est lemmatisé donc les adjectifs sont au masculin
# singulier (`social`, `européen`, `commun`...).

from collections import Counter

WHITELIST = [
    # --- Concepts politiques ---
    'sécurité_social',
    'service_public',
    'pouvoir_achat',
    'marché_commun',
    'liberté_enseignement', 'liberté_individuel', 'liberté_expression',
    'égalité_chance', 'égalité_fraternité',
    'droit_homme', 'droit_femme', 'droit_grève', 'droit_vote',
    'classe_ouvrier', 'classe_moyen',
    # --- Institutions ---
    'collectivité_local', 'collectivité_territorial',
    'fonction_publique',
    'enseignement_supérieur', 'enseignement_privé', 'enseignement_public',
    # --- Personnages ---
    'général_gaulle', 'giscard_estaing', 'arlette_laguiller',
    # --- Thèmes structurants ---
    'peine_mort',
    'code_nationalité', 'carte_séjour',
    'traité_rome', 'union_européen', 'construction_européen',
    'temps_travail', 'durée_travail', 'condition_travail',
    'salaire_minimum',
    'logement_social', 'protection_social',
    # --- Géographique (occurrences fortes) ---
    'alpes_maritime', 'franche_comté',
]

# Vérification : on compte chaque bigramme dans le corpus d'origine
all_docs = [doc for year in YEARS for doc in corpora[year]]
bigram_counts = Counter()
for doc in all_docs:
    for i in range(len(doc) - 1):
        bg = f'{doc[i]}_{doc[i+1]}'
        if bg in set(WHITELIST):
            bigram_counts[bg] += 1

MIN_OCCUR = 50
print(f'Vérification de la whitelist ({len(WHITELIST)} entrées) :\n')
print(f'{"Bigramme":<32} {"Occurrences":>12}   Statut')
print('-' * 60)
applied = []
for bg in WHITELIST:
    n = bigram_counts.get(bg, 0)
    status = '✓ retenu' if n >= MIN_OCCUR else f'⚠ rejeté (<{MIN_OCCUR})'
    print(f'  {bg:<30} {n:>12,}   {status}')
    if n >= MIN_OCCUR:
        applied.append(bg)
print(f'\n→ {len(applied)}/{len(WHITELIST)} bigrammes effectivement appliqués')

# Application sur les corpora (substitution gauche-droite, non-overlapping)
applied_set = set(applied)

def apply_bigrams(doc):
    out, i = [], 0
    while i < len(doc):
        if i + 1 < len(doc):
            bg = f'{doc[i]}_{doc[i+1]}'
            if bg in applied_set:
                out.append(bg)
                i += 2
                continue
        out.append(doc[i])
        i += 1
    return out

print('\nApplication aux corpora :')
for year in YEARS:
    n_before = sum(len(d) for d in corpora[year])
    corpora[year] = [apply_bigrams(doc) for doc in corpora[year]]
    n_after = sum(len(d) for d in corpora[year])
    n_bg = sum(1 for doc in corpora[year] for tok in doc if '_' in tok)
    print(f'  {year} : {n_before:,} -> {n_after:,} tokens  ({n_bg:,} occurrences de bigrammes)')

    with open(f'../data/processed/corpus_bigrams_{year}.pkl', 'wb') as f:
        pickle.dump(corpora[year], f)

Vérification de la whitelist (38 entrées) :

Bigramme                          Occurrences   Statut
------------------------------------------------------------
  sécurité_social                       3,797   ✓ retenu
  service_public                        2,753   ✓ retenu
  pouvoir_achat                             0   ⚠ rejeté (<50)
  marché_commun                           683   ✓ retenu
  liberté_enseignement                  1,321   ✓ retenu
  liberté_individuel                      734   ✓ retenu
  liberté_expression                      181   ✓ retenu
  égalité_chance                          990   ✓ retenu
  égalité_fraternité                    1,264   ✓ retenu
  droit_homme                             663   ✓ retenu
  droit_femme                             839   ✓ retenu
  droit_grève                             288   ✓ retenu
  droit_vote                                0   ⚠ rejeté (<50)
  classe_ouvrier                          483   ✓ retenu
  classe_moyen               

In [ ]:
# Entraînement avec initialisation t depuis t-1 (Kim et al. 2014)
# Méthode : on construit le vocab du nouveau corpus, puis on copie les vecteurs
# des mots déjà vus dans le modèle précédent.
models = {}
prev_model = None

for year in YEARS:
    print(f'\nEntraînement {year}...')
    corpus = corpora[year]

    model = Word2Vec(**PARAMS)
    model.build_vocab(corpus)

    if prev_model is not None:
        n_init = 0
        for word in model.wv.index_to_key:
            if word in prev_model.wv:
                model.wv.vectors[model.wv.key_to_index[word]] = prev_model.wv[word]
                n_init += 1
        print(f'  → {n_init}/{len(model.wv):,} mots initialisés depuis {YEARS[YEARS.index(year)-1]}')

    model.train(corpus, total_examples=model.corpus_count, epochs=model.epochs)
    model.save(f'../results/models/w2v_{year}.model')
    models[year] = model
    prev_model = model

    print(f'  ✅ {year} — vocab: {len(model.wv):,} mots')

print('\n✅ Tous les modèles entraînés et sauvegardés.')


Entraînement 1973...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

  ✅ 1973 — vocab: 3,787 mots

Entraînement 1978...
  → 3232/4,353 mots initialisés depuis 1973


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

  ✅ 1978 — vocab: 4,353 mots

Entraînement 1981...
  → 2771/2,965 mots initialisés depuis 1978


...

  ✅ 1981 — vocab: 2,965 mots

Entraînement 1988...
  → 2210/2,651 mots initialisés depuis 1981


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  ✅ 1988 — vocab: 2,651 mots

Entraînement 1993...
  → 2440/4,066 mots initialisés depuis 1988


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  ✅ 1993 — vocab: 4,066 mots

✅ Tous les modèles entraînés et sauvegardés.


In [24]:
MOTS_CIBLES = ['liberté', 'peuple', 'nation', 'europe', 'sécurité', 'travail']

for mot in MOTS_CIBLES:
    print(f'\n=== {mot.upper()} ===')
    for year in YEARS:
        m = models[year]
        if mot in m.wv:
            voisins = [w for w, _ in m.wv.most_similar(mot, topn=5)]
            print(f'  {year}: {voisins}')
        else:
            print(f'  {year}: mot absent du vocabulaire')


=== LIBERTÉ ===
  1973: ['liberté_individuel', 'démocratie', 'libertés', 'respect', 'soucieux']
  1978: ['liberté_enseignement', 'entreprendre', 'justice', 'libertés', 'progrès']
  1981: ['libre', 'liberté_enseignement', 'entreprendre', 'progrès', 'respect']
  1988: ['justice', 'paix', 'valeur', 'attacher', 'tolérance']
  1993: ['égalité_fraternité', 'respect', 'justice', 'total', 'paix']

=== PEUPLE ===
  1973: ['ministre', 'impérialisme', 'hexagone', 'solidarité', 'dernier']
  1978: ['représentant', 'souveraineté', 'réalité', 'œuvre', 'monde']
  1981: ['paix', 'engager', 'gouvernement', 'capable', 'solidarité']
  1988: ['idée', 'préoccuper', 'appareil', 'promesse', 'carrière']
  1993: ['autoflagellation', 'europe', 'opprimé', 'confédéral', 'paix']

=== NATION ===
  1973: ['convaincre', 'démagogie', 'pourtant', 'france', 'vital']
  1978: ['supranational', 'indépendance', 'fort', 'américain', 'puissance']
  1981: ['grandeur', 'histoire', 'france', 'monde', 'solidarité']
  1988: ['fort